<a href="https://colab.research.google.com/github/yugan243/Qdrant-Essentials/blob/main/Day_02_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project HNSW Performance Benchmarking

In [1]:
!pip install -q qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 9.7 MB/s eta 0:00:00


### 1. Prepare the environment

In [2]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from google.colab import userdata
import time, os, numpy as np
from datasets import load_dataset

### 2. Create the Qdrant Client

In [3]:
client = QdrantClient(
    api_key=userdata.get('QDRANT_API_KEY'),
    url=userdata.get('QDRANT_URL')
)

encoder = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### 3. Create multiple test collections

In [4]:
configs = [
    {"name": "fast_initial_upload", "m": 0, "ef_construct": 100},
    {"name": "memory_optimized", "m": 8, "ef_construct": 100},
    {"name": "balanced", "m": 16, "ef_construct": 200},
    {"name": "high_quality", "m": 32, "ef_construct": 400}
]

for config in configs:
    collection_name = f"my_domain_{config["name"]}"
    if client.collection_exists(collection_name=collection_name):
        client.delete_collection(collection_name=collection_name)

    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=1536,
            distance=models.Distance.COSINE
        ),
        hnsw_config=models.HnswConfigDiff(
            m=config["m"],
            ef_construct=config["ef_construct"],
            full_scan_threshold=10  # Force HNSW instead of full scan
        ),
        optimizers_config=models.OptimizersConfigDiff(
            indexing_threshold=10 # Force indexing even on small sets
        )
    )

    print(f"Created collection: {collection_name}")


Created collection: my_domain_fast_initial_upload
Created collection: my_domain_memory_optimized
Created collection: my_domain_balanced
Created collection: my_domain_high_quality


### 4. Prepare the Dataset

In this project we use the same data set we use for the demo. "Qdrant/dbpedia-entities-openai3-text-embedding-3-large-1536-100K". But this dataset consists of 100k data. Since that would consume a lot of resources as we create 4 different collections for this project, we take the first 10k rows from the dataset and prepare it for upload.

In [5]:
print("Loading DB Pedia 100k dataset....")
ds = load_dataset("Qdrant/dbpedia-entities-openai3-text-embedding-3-large-1536-100K")


Loading DB Pedia 100k dataset....


README.md:   0%|          | 0.00/420 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/318M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/318M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/318M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [6]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['_id', 'title', 'text', 'text-embedding-3-large-1536-embedding'],
        num_rows: 100000
    })
})


In [7]:
# Take only 10k dataset
rows_10k = ds['train'].select(range(10000))

# Convert to a list
data = rows_10k.to_list()

### 5. Measure Upload Performance for each Configuration

In [10]:
def upload_with_timing(collection_name, data, config_name):

    # To store the points going to upload
    points = []

    for i, item in enumerate(data):
        points.append(models.PointStruct(
            id=i,
            vector=item['text-embedding-3-large-1536-embedding'],
            payload={
                "_id": item['_id'],
                "title": item['title'],
                "text": item['text'],
                "length": len(item['text']),
                "word_count": len(item['text'].split()),
                "has_numbers": any(char.isdigit() for char in item['text']),
                "has_keywords": any(
                    keyword in item['text'].lower() for keyword in ['important', 'key', 'main']
                )

            }
        ))

    # Warm up (Waking up the internal componenets of the remote server)
    client.query_points(collection_name=collection_name, query=points[0].vector, limit=1)

    start_time = time.time()
    client.upload_points(collection_name=collection_name, points=points)
    upload_time = time.time() - start_time

    print(f"{config_name}: Uploaded {len(points)} points in {upload_time:.2f}s")
    return upload_time





In [11]:
# Upload points to each collection
upload_times = {}

for config in configs:
    collection_name = f"my_domain_{config["name"]}"
    upload_times[config["name"]] = upload_with_timing(collection_name=collection_name, data=data, config_name=config["name"])

fast_initial_upload: Uploaded 10000 points in 37.51s
memory_optimized: Uploaded 10000 points in 34.03s
balanced: Uploaded 10000 points in 33.40s
high_quality: Uploaded 10000 points in 34.50s


In [16]:
from qdrant_client.grpc import CollectionStatus

def wait_for_indexing(collection_name, timeout=60, poll_interval=1):
    print(f"Waiting for collection '{collection_name}' to be indexed...")
    start_time = time.time()

    while time.time() - start_time < timeout:
        info = client.get_collection(collection_name=collection_name)
        if info.indexed_vectors_count > 0 and info.status == models.CollectionStatus.GREEN:
            print(f"Success! Collection {collection_name} is indexed and ready.")
            print(f"- Status: {info.status.value}")
            print(f"- Indexed vector count: {info.indexed_vectors_count}")
            return

        print(f"- Status: {info.status.value}, Indexed vectors: {info.indexed_vectors_count}. Waiting...")
        time.sleep(poll_interval)

    info = client.get_collection(collection_name=collection_name)

    raise Exception(
        f"Timeout reached after {timeout} seconds. Collection {collection_name} is not ready. "
        f"Final status: {info.status.value}, Indexed vectors: {info.indexed_vectors_count}"
    )




In [17]:
for config in configs:
    if config["m"] > 0:
        collection_name = f"my_domain_{config["name"]}"
        wait_for_indexing(collection_name=collection_name)

Waiting for collection 'my_domain_memory_optimized' to be indexed...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vect

### 6. Benchmark Search Performance

In [18]:
def benchmark_search(collection_name, query_embedding, ef_values=[64, 128, 256]):
    # Warm up
    client.query_points(collection_name=collection_name, query=query_embedding, limit=1)

    #hnsw_ef: higher = better recall but slower.
    results = {}
    for hnsw_ef in ef_values:
        times = []

        # Run multiple queries for more reliable timing
        for _ in range(25):
            start_time = time.time()

            _ = client.query_points(
                collection_name=collection_name,
                query=query_embedding,
                limit=10,
                search_params=models.SearchParams(hnsw_ef=hnsw_ef),
                with_payload=False
            )
            times.append((time.time() - start_time) * 1000)

        results[hnsw_ef] = {
            "avg_time": np.mean(times),
            "min_time": np.min(times),
            "max_time": np.max(times),
        }

    return results




In [19]:
# Test using a Query

import requests

url = "https://storage.googleapis.com/qdrant-examples/query_embedding_day_2.json"
resp = requests.get(url)

test_query = resp.json()["query_vector"]

print(f"Embedding dimensions: {len(test_query)}")
print(f"First 5 values: {test_query[:5]}")

Embedding dimensions: 1536
First 5 values: [-0.012367082759737968, -0.01300355140119791, -0.003559330478310585, -0.0013035375159233809, 0.024401243776082993]


In [20]:
performance_results = {}
for config in configs:
    if config["m"] > 0: # Skip m=0 collections for search
        collection_name = f"my_domain_{config["name"]}"
        performance_results[config["name"]] = benchmark_search(
            collection_name=collection_name,
            query_embedding=test_query
        )



In [21]:
performance_results

{'memory_optimized': {64: {'avg_time': np.float64(82.75312423706055),
   'min_time': np.float64(72.3719596862793),
   'max_time': np.float64(96.00687026977539)},
  128: {'avg_time': np.float64(73.75642776489258),
   'min_time': np.float64(71.86484336853027),
   'max_time': np.float64(79.94294166564941)},
  256: {'avg_time': np.float64(74.22168731689453),
   'min_time': np.float64(72.57843017578125),
   'max_time': np.float64(80.8873176574707)}},
 'balanced': {64: {'avg_time': np.float64(72.9813289642334),
   'min_time': np.float64(71.79570198059082),
   'max_time': np.float64(77.5597095489502)},
  128: {'avg_time': np.float64(75.85641860961914),
   'min_time': np.float64(72.6168155670166),
   'max_time': np.float64(86.22288703918457)},
  256: {'avg_time': np.float64(76.0323429107666),
   'min_time': np.float64(73.45032691955566),
   'max_time': np.float64(80.50727844238281)}},
 'high_quality': {64: {'avg_time': np.float64(75.48783302307129),
   'min_time': np.float64(72.6621150970459),

Balanced configuration gives the best results (m=16, ef_construct=200)

### 7. Measure the Payload indexing impact

In [34]:
def test_filtering_performance(collection_name, query_embedding):

    # Test filter without index
    filter_condition = models.Filter(
        must=[models.FieldCondition(key="length", range=models.Range(gte=10, lte=200))]
    )

    # Demo only: unindexed_filtering_retrieve=True forces a scan; we turn it off right after measuring.
    client.update_collection(
        collection_name=collection_name,
        strict_mode_config=models.StrictModeConfig(unindexed_filtering_retrieve=True),
    )

    # Warm up
    client.query_points(collection_name=collection_name, query=query_embedding, limit=10)

    # Timing without payload index
    times = []
    for _ in range(25):
        start_time = time.time()
        _ = client.query_points(
            collection_name=collection_name,
            query=query_embedding,
            query_filter=filter_condition,
            limit=10,
            with_payload=False
        )
        times.append((time.time() - start_time) * 1000)
    time_without_index = np.mean(times)


    # Create a payload index
    client.create_payload_index(
        collection_name=collection_name,
        field_name="length",
        field_schema=models.PayloadSchemaType.INTEGER,
        wait=True
    )

    # HNSW was already built; adding the payload index doesn’t rebuild it.
    # Bump ef_construct (+1) once to trigger a safe rebuild.
    base_ef = client.get_collection(
        collection_name=collection_name
    ).config.hnsw_config.ef_construct
    new_ef_construct = base_ef + 1

    client.update_collection(
        collection_name=collection_name,
        hnsw_config=models.HnswConfigDiff(ef_construct=new_ef_construct),
        strict_mode_config=models.StrictModeConfig(unindexed_filtering_retrieve=False)
    )

    wait_for_indexing(collection_name=collection_name, timeout=300)

    # Warm up
    client.query_points(collection_name, query=query_embedding, limit=1)

    # Timing without payload index
    times = []
    for _ in range(25):
        start_time = time.time()
        _ = client.query_points(
            collection_name=collection_name,
            query=query_embedding,
            query_filter=filter_condition,
            limit=10,
            with_payload=False
        )
        times.append((time.time() - start_time) * 1000)
    time_with_index = np.mean(times)

    return {
        "without_index": time_without_index,
        "with_index": time_with_index,
        "speed up": time_without_index/time_with_index
    }


In [35]:
best_collection = "my_domain_balanced" # Based on the results
filtering_results = test_filtering_performance(best_collection, test_query)

Waiting for collection 'my_domain_balanced' to be indexed...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 10000. Waiting...
- Status: yellow, Indexed vectors: 100

In [37]:
print("=" * 60)
print("PERFORMANCE OPTIMIZATION RESULTS")
print("=" * 60)

print("\n1) Upload Performance:")
for config_name, time_taken in upload_times.items():
    print(f"   {config_name}: {time_taken:.2f}s")

print("\n2) Search Performance (hnsw_ef=128):")
for config_name, results in performance_results.items():
    if 128 in results:
        print(f"   {config_name}: {results[128]['avg_time']:.2f}ms")

print("\n3) Filtering Impact:")
print(f"   Without index: {filtering_results['without_index']:.2f}ms")
print(f"   With index: {filtering_results['with_index']:.2f}ms")
print(f"   Speedup: {filtering_results['speed up']:.1f}x")

PERFORMANCE OPTIMIZATION RESULTS

1) Upload Performance:
   fast_initial_upload: 37.51s
   memory_optimized: 34.03s
   balanced: 33.40s
   high_quality: 34.50s

2) Search Performance (hnsw_ef=128):
   memory_optimized: 73.76ms
   balanced: 75.86ms
   high_quality: 76.63ms

3) Filtering Impact:
   Without index: 81.71ms
   With index: 83.59ms
   Speedup: 1.0x


The dataset is not sifficient for visualizing the real impacts of differenct configurations and also the payload indexing. I used 10k dataset because of the Qdrant storage limitations. in future works will try to increae the size of the dataset.